# Notebook 03 — Lógica Paracompleta aplicada à base de diabetes

Neste notebook, vamos trabalhar com uma ideia muito importante:

> Ausência de informação não é a mesma coisa que informação negativa.

Em saúde, isso é essencial.

Se um prontuário não informa se o paciente tem poliúria, isso não significa que ele não tenha poliúria.  
Significa apenas que não sabemos.

A lógica paracompleta é útil para preservar essa lacuna.

## Etapa 1 — Enviar o arquivo da base

In [ ]:
# ============================================================
# 1. Faça upload do arquivo diabetes_data_upload.csv
# ============================================================

from google.colab import files
import io
import pandas as pd

print("Selecione o arquivo diabetes_data_upload.csv no seu computador.")
uploaded = files.upload()

# Pega automaticamente o primeiro arquivo enviado
nome_arquivo = list(uploaded.keys())[0]

df = pd.read_csv(io.BytesIO(uploaded[nome_arquivo]))

print("Arquivo carregado com sucesso!")
print(f"Nome do arquivo: {nome_arquivo}")
print(f"Quantidade de linhas: {df.shape[0]}")
print(f"Quantidade de colunas: {df.shape[1]}")

df.head()

## Etapa 2 — Preparar funções auxiliares

In [ ]:
# ============================================================
# Bibliotecas e funções auxiliares
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Coluna-alvo da base
TARGET = "class"

# Marcadores clínico-sintomáticos presentes na base
SINTOMAS = [
    "Polyuria",
    "Polydipsia",
    "sudden weight loss",
    "weakness",
    "Polyphagia",
    "Genital thrush",
    "visual blurring",
    "Itching",
    "Irritability",
    "delayed healing",
    "partial paresis",
    "muscle stiffness",
    "Alopecia",
    "Obesity",
]

# Sintomas escolhidos como centrais para simular lacunas ou conflito
# A escolha é didática: são sintomas que aparecem com relevância clínica
# e ajudam a mostrar o comportamento das lógicas.
SINTOMAS_CENTRAIS = [
    "Polyuria",
    "Polydipsia",
    "sudden weight loss",
    "weakness",
    "Polyphagia",
    "partial paresis",
]

# Pesos heurísticos.
# Eles não foram aprendidos por machine learning.
# São apenas uma forma didática de dizer:
# "alguns sintomas pesam mais na suspeita do que outros".
PESOS = {
    "Polyuria": 3.0,
    "Polydipsia": 3.0,
    "sudden weight loss": 2.0,
    "weakness": 1.0,
    "Polyphagia": 1.5,
    "Genital thrush": 0.5,
    "visual blurring": 1.0,
    "Itching": 0.25,
    "Irritability": 1.0,
    "delayed healing": 0.75,
    "partial paresis": 2.0,
    "muscle stiffness": 0.5,
    "Alopecia": 0.25,
    "Obesity": 0.25,
}

def sim_nao_para_numero(valor):
    '''
    Converte respostas categóricas da base para números.

    Yes -> 1
    No  -> 0
    NaN -> NaN

    Essa conversão permite que a lógica opere sobre evidências.
    '''
    if pd.isna(valor):
        return np.nan

    valor = str(valor).strip().lower()

    if valor == "yes":
        return 1.0
    elif valor == "no":
        return 0.0
    else:
        return np.nan

def pertinencia_idade(idade):
    '''
    Transforma idade em uma evidência gradual simples.

    Menos de 30 anos: pertinência 0
    Entre 30 e 60 anos: crescimento linear
    Acima de 60 anos: pertinência 1

    Não é uma regra médica definitiva.
    É apenas uma simplificação didática para mostrar como
    variáveis numéricas podem entrar no raciocínio lógico.
    '''
    if pd.isna(idade):
        return 0.5

    if idade < 30:
        return 0.0
    elif idade > 60:
        return 1.0
    else:
        return (idade - 30) / 30

def classe_para_binario(valor):
    '''
    Converte a classe original da base para comparação simples.

    Positive -> 1
    Negative -> 0
    '''
    return 1 if valor == "Positive" else 0

def mostrar_distribuicao_decisoes(resultado, coluna="decisao", titulo="Distribuição das decisões"):
    '''
    Plota um gráfico simples com a distribuição das saídas produzidas pela lógica.
    '''
    contagem = resultado[coluna].value_counts()

    plt.figure(figsize=(8, 4))
    barras = plt.bar(contagem.index.astype(str), contagem.values)
    plt.title(titulo)
    plt.xlabel("Saída produzida pela lógica")
    plt.ylabel("Quantidade de registros")
    plt.xticks(rotation=25, ha="right")

    for barra in barras:
        altura = barra.get_height()
        plt.text(
            barra.get_x() + barra.get_width()/2,
            altura,
            str(int(altura)),
            ha="center",
            va="bottom"
        )

    plt.tight_layout()
    plt.show()

def resumo_simples(resultado, coluna_decisao="decisao"):
    '''
    Gera um resumo didático da saída da lógica.

    A ideia não é tratar a lógica como um classificador tradicional,
    mas observar como ela decide, quando se abstém e como distribui
    suas respostas.
    '''
    total = len(resultado)

    resumo = resultado[coluna_decisao].value_counts().reset_index()
    resumo.columns = ["decisao", "quantidade"]
    resumo["percentual"] = 100 * resumo["quantidade"] / total

    return resumo

## Etapa 3 — Conhecer a base

In [ ]:
# ============================================================
# Olhando a base antes de aplicar qualquer lógica
# ============================================================

print("Dimensão da base:", df.shape)
print("\nColunas disponíveis:")
print(df.columns.tolist())

print("\nDistribuição da classe:")
display(df[TARGET].value_counts().reset_index().rename(columns={"index": "classe", TARGET: "quantidade"}))

print("\nQuantidade de valores ausentes por coluna:")
display(df.isna().sum().reset_index().rename(columns={"index": "coluna", 0: "ausentes"}))

print("\nResumo da idade:")
display(df["Age"].describe())

## Etapa 4 — Ideia da lógica paracompleta

A lógica paracompleta permite que uma proposição não seja nem confirmada nem negada.

Na prática deste notebook:

- se há informação suficiente, o sistema decide;
- se faltam sintomas centrais, o sistema pode responder `inconclusivo`;
- a ausência de dado não será tratada como `No`.

Isso evita uma inferência precipitada.

In [ ]:
def decisao_paracompleta(linha):
    lacunas_centrais = sum(pd.isna(linha[coluna]) for coluna in SINTOMAS_CENTRAIS)
    lacunas_totais = sum(pd.isna(linha[coluna]) for coluna in SINTOMAS)

    valores_observados = []
    pesos_observados = []

    for sintoma in SINTOMAS:
        valor = sim_nao_para_numero(linha[sintoma])

        # Aqui está o ponto paracompleto:
        # se o valor está ausente, ele simplesmente não entra como negativo.
        if not pd.isna(valor):
            valores_observados.append(valor)
            pesos_observados.append(PESOS[sintoma])

    if len(valores_observados) == 0:
        return pd.Series({
            "escore_paracompleto": np.nan,
            "lacunas_centrais": lacunas_centrais,
            "lacunas_totais": lacunas_totais,
            "decisao": "inconclusivo"
        })

    valores_observados.append(pertinencia_idade(linha["Age"]))
    pesos_observados.append(0.75)

    escore = np.average(valores_observados, weights=pesos_observados)

    if lacunas_centrais >= 2:
        decisao = "inconclusivo"
    elif lacunas_centrais == 1 and 0.35 <= escore < 0.65:
        decisao = "suspeita_com_evidencia_incompleta"
    elif escore >= 0.65:
        decisao = "suspeita_alta"
    elif escore >= 0.35:
        decisao = "suspeita_moderada"
    else:
        decisao = "suspeita_baixa"

    return pd.Series({
        "escore_paracompleto": escore,
        "lacunas_centrais": lacunas_centrais,
        "lacunas_totais": lacunas_totais,
        "decisao": decisao
    })

def aplicar_logica_paracompleta(base):
    resultado = base.copy()
    saidas = resultado.apply(decisao_paracompleta, axis=1)
    resultado = pd.concat([resultado, saidas], axis=1)
    resultado["classe_binaria"] = resultado[TARGET].apply(lambda x: 1 if x == "Positive" else 0)
    return resultado

## Etapa 5 — Aplicar a lógica paracompleta na base original

Como a base original não tem lacunas, a lógica tende a decidir com mais frequência.

Mesmo assim, a estrutura já está preparada para lidar com ausência de dados.

In [ ]:
resultado_original = aplicar_logica_paracompleta(df)

display(resultado_original[[
    "Age", "Gender", "Polyuria", "Polydipsia",
    "sudden weight loss", "weakness",
    "class", "escore_paracompleto",
    "lacunas_centrais", "lacunas_totais", "decisao"
]].head(10))

display(resumo_simples(resultado_original))
mostrar_distribuicao_decisoes(
    resultado_original,
    titulo="Lógica paracompleta — base original"
)

## Etapa 6 — Simular dados incompletos

Agora vamos criar lacunas artificiais em sintomas centrais.

Esse é o cenário em que a lógica paracompleta mostra melhor sua função.

In [ ]:
# ============================================================
# Criando uma cópia da base com lacunas simuladas
# ============================================================

def criar_base_incompleta(base, taxa_lacunas=0.20, semente=42):
    '''
    Cria uma versão da base com alguns sintomas centrais apagados.

    Isso simula uma situação comum em saúde:
    nem toda informação foi registrada, perguntada ou confirmada.
    '''
    base_incompleta = base.copy()
    rng = np.random.default_rng(semente)

    for coluna in SINTOMAS_CENTRAIS:
        mascara = rng.random(len(base_incompleta)) < taxa_lacunas
        base_incompleta.loc[mascara, coluna] = np.nan

    return base_incompleta

df_incompleta = criar_base_incompleta(df, taxa_lacunas=0.20)

print("Base original:")
display(df[SINTOMAS_CENTRAIS].head())

print("Base com lacunas simuladas:")
display(df_incompleta[SINTOMAS_CENTRAIS].head())

print("Valores ausentes após simulação:")
display(df_incompleta[SINTOMAS_CENTRAIS].isna().sum())

In [ ]:
resultado_incompleto = aplicar_logica_paracompleta(df_incompleta)

display(resultado_incompleto[[
    "Age", "Gender", "Polyuria", "Polydipsia",
    "sudden weight loss", "weakness",
    "class", "escore_paracompleto",
    "lacunas_centrais", "lacunas_totais", "decisao"
]].head(10))

display(resumo_simples(resultado_incompleto))
mostrar_distribuicao_decisoes(
    resultado_incompleto,
    titulo="Lógica paracompleta — base com lacunas"
)

## Etapa 7 — Comparar quantidade de inconclusivos

A saída `inconclusivo` não é erro.

Ela representa uma decisão cautelosa:  
o sistema reconhece que não tem informação suficiente.

In [ ]:
print("Base original:")
display(resumo_simples(resultado_original))

print("Base com lacunas:")
display(resumo_simples(resultado_incompleto))

## Etapa 8 — Reflexão

A lógica paracompleta é útil quando o maior risco é decidir com pouca informação.

Ela ajuda a evitar frases perigosas como:

> O dado não está registrado, então o sintoma não existe.

Em sistemas médicos, essa distinção é fundamental.

In [ ]:
resultado_original.to_csv("resultado_paracompleta_original.csv", index=False)
resultado_incompleto.to_csv("resultado_paracompleta_incompleta.csv", index=False)

print("Arquivos exportados:")
print("- resultado_paracompleta_original.csv")
print("- resultado_paracompleta_incompleta.csv")